# Traces of Joy: Exploring Urban Happiness through Machine Vision and Human Feeling

**MUSA 650 — Final Project**

**Group members:** Fan Yang, Zhiyuan Zhao, Qiushi Yu

---

## 1. Problem Definition & Technical Justification

### 1.1 Where Does a City Feel Happy?

Where in a city will people feel happy? Conventional planning practice is fluent in the language of walkability scores, traffic volumes, and property values, but almost silent on the emotional dimension of place. A neighborhood can score well on every standard metric and still feel joyless to the people who live in it — and conversely, ordinary street corners with no special claim on any index can be the places someone returns to, again and again, because they make the day feel lighter.

This project begins from a participatory study conducted by the Happiness Lab at Drexel University (Zillmer, 2025). In that study, **243 undergraduate students each identified one location in Philadelphia where they felt genuinely happy**. Their answers clustered into **28 representative happy places**, and the variety is telling. Some are public squares you would expect — Rittenhouse Square, the steps of the Philadelphia Museum of Art. Others are quieter and more idiosyncratic: the cat park in Mantua, the cozy corner of Maison Sweet where you might stay longer than planned, the rooftop at Cira Green, a bench on the Schuylkill Banks. **None of these are tourist destinations.** They are ordinary places where positive emotional experiences arise in the middle of daily life.

Our question is the inverse: can we look at the visual and social fabric of these 28 places, and use what we find to identify *other* places across Philadelphia that are likely to feel the same way?

### 1.2 Who This Is For

The intended users are **city planners and community organizations**. Their practical need is to identify which street-level environments are most strongly associated with happiness, so that decisions around mixed-use zoning, public space investment, and streetscape improvement can be guided by emotional evidence alongside the usual traffic and tax-base metrics.

### 1.3 What the Model Delivers

A **happiness probability score for every street-level sampling point** in the city — roughly 46,000 points placed every 200 m along Philadelphia's road network. Stitched together, these scores form a continuous spatial surface that shows how the visual form of the city relates to subjective well-being: a map of where the ingredients of happy places already exist, where they almost exist, and where they are missing entirely.

### 1.4 What the Model Is Actually Learning

The model learns to identify patterns resembling known happy places within a **14-dimensional feature space**:

- **6 visual features** from semantic segmentation of GSV panoramas — sky ratio, green view index, building ratio, road ratio, vehicle ratio, person ratio.
- **8 socioeconomic features** from 2022 ACS 5-year estimates — median income, poverty rate, college-plus education rate, percent white, median age, owner-occupancy rate, unemployment rate, median home value.

Crucially, the model does **not** detect happiness directly. Its predictions represent *environmental similarity to confirmed happy places*, consistent with Naik et al. (2017), who treat computer vision outputs as correlates of urban conditions rather than causal explanations. We are not measuring joy; we are mapping the places that statistically resemble where joy has already been reported.

### 1.5 Why PU Learning Is the Right Task Type

This is a classic **Positive-Unlabeled (PU) Learning** problem: we know where people *felt* happy, but we have no verified "unhappy" locations. Treating all ~46,000 unlabeled points as negatives would introduce severe labeling bias — many of them may be undiscovered happy places. Standard binary classification would penalize exactly that kind of undiscovered positive; regression requires continuous happiness scores that the data does not provide.

The **two-stage PU framework** addresses this directly: (i) compute the 28-point centroid in standardized feature space and select the 30% of unlabeled points *farthest* from the centroid as reliable negatives; (ii) train classifiers on the resulting pseudo-labeled dataset.

### 1.6 Anticipated Failure Modes and What We Observed

| Anticipated | Observed in practice |
|-------------|---------------------|
| Statistical instability from 28 positives | Cross-validation AUC standard deviation was non-trivial (±0.017 – 0.022), but both models still cleanly exceeded the 0.85 target. Stratified 5-fold CV contained 5–6 validation positives per fold as expected. |
| Sampling bias toward Center City and University City | Confirmed: `pct_white` appears as a significant positive predictor (β=+0.52, p<0.001), reflecting the demographic composition of the undergraduate respondent pool rather than a universal predictor of happiness. |
| Hidden garden problem (GSV cannot photograph rooftop/enclosed spaces) | Confirmed: `green_view_index` is **not significant** (β=−0.04, p=0.44). This is striking given the urban-perception literature; we attribute it to several happy places (Cira Green rooftop, John F. Collings Park) whose visual features are captured by nearby road-level images rather than the spaces themselves. |
| Ecological fallacy from tract-level Census | Unobservable without individual-level ground truth; acknowledged as a structural limitation. |
| Association vs. causation | The owner-occupancy paradox (strongest negative predictor, β=−2.06) exemplifies this: we cannot distinguish whether low owner-occupancy *causes* happiness clustering or whether both arise from shared underlying conditions (walkability, mixed use). |

---

## 2. Methodological Precedent

Four sources shaped the design decisions; the reading evolved during implementation as we reconciled conflicting guidance on feature aggregation and class imbalance.

**Li & Ratti (2019) — *Using Google Street View for street-level urban form analysis*.** Establishes regular-interval road-centerline sampling and Otsu's thresholding on sky pixels to estimate the Sky View Factor in Cambridge, MA. We adopt the same sampling strategy at 200 m intervals (their 100 m was infeasible at our city scale with API quota constraints) and use Otsu's thresholding as our non-ML baseline, applied to the green view index rather than sky.

**Li et al. (2015) — *A modified green view index using Google Street View*.** Defines the Green View Index as the pixel proportion of vegetation in GSV panoramas and validates it across cities. We use GVI as one of six visual features and as the baseline's sole input. Their caveats about seasonal variation are relevant here: Philadelphia GSV imagery spans 2022–2024, so leaf-off versus leaf-on conditions inject noise we cannot control for.

**Naik et al. (2017) — *Computer vision uncovers predictors of physical urban change*.** Links street-level visual features to socioeconomic characteristics via spatially matched Census data and evaluates with Kendall's tau. Their core finding — that street appearance predicts social conditions — is the methodological foundation for our integration of GSV-derived features with ACS variables. We follow their framing of CV outputs as correlates rather than causal explanations.

**Xie et al. (2021) — *SegFormer*.** Transformer-based semantic segmentation achieving strong performance on ADE20K and Cityscapes. The lightweight B0 variant balances accuracy with GPU efficiency; its hierarchical encoder (no positional encoding) handles our variable-resolution panoramas gracefully. We apply the ADE20K-pretrained SegFormer-B0 to all GSV panoramas via GPU-accelerated batch processing, then consolidate the 150 ADE20K classes into six urban feature dimensions.

---

## 3. Data & Preprocessing

### 3.1 Data Sources

| Data | Description | Source | Resolution / Coverage |
|------|-------------|--------|-----------------------|
| GSV panoramas | 3328×1664 px, four-directional, collected 2022–2024 | Google Maps Street View API | ~46,000 points along road centerlines, 200 m spacing |
| Happy places | 28 student-identified points with descriptions | Drexel Happiness Lab (Zillmer, 2025) | Point features within Philadelphia |
| Road centerlines | Street centerlines for all accessible roads | OpenDataPhilly | NAD 1983 / PA South (ft) |
| Census | 2022 ACS 5-year, 384 Philadelphia tracts, 8 socioeconomic variables | U.S. Census Bureau API | Tract level (~4,100 residents/tract) |

### 3.2 Sampling Point Generation

Road centerlines were projected to NAD 1983 / Pennsylvania South (ft) and clipped to the Philadelphia boundary. Sampling points were interpolated every 200 m along each line (treating `LineString` and `MultiLineString` geometries separately to handle disconnected segments), then merged with the 28 happy points into a single shapefile of 46,186 records.

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")

df = pd.read_csv(DATA / "analysis_data" / "all_points_full.csv")
print(f"Total sampling points: {len(df):,}")
print(f"  Labeled happy points: {int(df['is_happy'].sum())}")
print(f"  Unlabeled road samples: {int((df['is_happy'] == 0).sum()):,}")

### 3.3 Data Quality Issues and Fixes

**Missing GSV imagery.** ~2% of sampling points had no Street View coverage (private roads, new developments, enclosed parks). These were excluded after logging; Mapillary was not needed as a fallback.

**Census API sentinel values.** The ACS API returns `-666666666` for suppressed small-population tracts. We replaced these with `NaN` and imputed missing values with the feature-wise median prior to modeling. In our cleaning pass, `median_income` had 2,615 sentinel values, `median_age` had 1,985, and `median_home_value` had 3,066 — non-trivial and would have poisoned the scaling if left in.

**Hidden garden spatial mismatch.** Several happy places (rooftop parks, internal courtyards) are not visible from public roads. GSV assigns the nearest accessible panorama, which may be tens or hundreds of meters away. We document this as a structural limitation rather than a fixable data issue.

### 3.4 Feature Engineering

**Semantic segmentation.** SegFormer-B0 (ADE20K-pretrained) was applied to all 46,000 panoramas via GPU batch processing. The 150 ADE20K classes were consolidated into six urban feature dimensions by pixel-proportion aggregation:

```
sky_ratio        ← class 2
green_view_index ← classes {4, 9, 17, 66, 72}  (tree, grass, plant, palm, flower)
building_ratio   ← classes {1, 25, 48, 84}     (building, house, skyscraper, tower)
road_ratio       ← classes {6, 11, 52}         (road, sidewalk, path)
vehicle_ratio    ← classes {20, 80, 83, 102, 127}  (car, bus, truck, van, motorcycle)
person_ratio     ← class 12                    (person)
```

**Segmentation quality check.** Because we use the pretrained model out-of-the-box without fine-tuning on Philadelphia imagery, we needed to verify that SegFormer-B0's output was reliable on our target scenes. We manually reviewed a stratified random sample of **200 segmented panoramas** — all 28 happy-place images plus 172 randomly selected road samples covering Center City, University City, Northeast Philadelphia, and South Philadelphia. For each image, we overlaid the segmentation mask on the source panorama and scored each of the six target classes on a three-point scale: *correct* (no major confusion), *partial* (small misclassifications in peripheral regions that do not materially change the pixel ratio), or *incorrect* (major confusion affecting the ratio).

| Class | Correct | Partial | Incorrect |
|-------|---------|---------|-----------|
| sky | 94% | 5% | 1% |
| vegetation (green view) | 86% | 11% | 3% |
| building | 92% | 7% | 1% |
| road | 93% | 6% | 1% |
| vehicle | 91% | 8% | 1% |
| person | 88% | 9% | 3% |

Most *partial* errors in the vegetation class came from dry grass or leafless shrubs being misclassified as ground, and most *partial* errors in the person class came from thin, partially occluded pedestrians being missed. Critically, we found **no systematic direction of error** that would bias the aggregated pixel ratios — over- and under-estimation appeared roughly balanced within each class. This audit gave us confidence that the ADE20K-pretrained model is adequate for our feature extraction without the expense (and overfitting risk, given 28 positives) of fine-tuning on our own data.

**Spatial join.** Census tract boundaries were downloaded for Pennsylvania (42), filtered to Philadelphia County (101), and merged with ACS data via `GEOID`. Each sampling point was assigned the tract it falls within.

**Final feature vector.** 6 visual + 8 socioeconomic = **14 dimensions**, standardized with `StandardScaler` prior to modeling.

In [ ]:
STREET_FEATURES = [
    "sky_ratio",
    "green_view_index",
    "building_ratio",
    "road_ratio",
    "vehicle_ratio",
    "person_ratio",
]
CENSUS_FEATURES = [
    "median_income",
    "poverty_rate",
    "pct_college",
    "pct_white",
    "median_age",
    "pct_owner_occupied",
    "unemployment_rate",
    "median_home_value",
]
ALL_FEATURES = STREET_FEATURES + CENSUS_FEATURES
print(f"Feature vector dimensionality: {len(ALL_FEATURES)}")

# Happy vs. other summary
happy = df[df["is_happy"] == 1]
other = df[df["is_happy"] == 0]
summary = pd.DataFrame(
    {
        "happy_mean": happy[ALL_FEATURES].mean(),
        "other_mean": other[ALL_FEATURES].mean(),
    }
).round(4)
summary["diff"] = (summary["happy_mean"] - summary["other_mean"]).round(4)
summary

---

## 4. Modeling Approach

### 4.1 Baseline: Otsu's Thresholding on Green View Index

Greenery is among the most consistently cited positive environmental indicators in urban perception research (Li et al., 2015). The baseline applies Otsu's thresholding to the green view index — a single-feature, non-machine-learning approach analogous to NDWI thresholding in remote sensing. Points above the automatically chosen threshold are predicted as happy.

**This is intentionally a weak baseline.** If greenery alone were sufficient, the multi-feature PU Learning approach would be unjustified.

In [ ]:
baseline = pd.read_csv(DATA / "modeling_results" / "baseline_otsu_results.csv").iloc[0]

print("Otsu Baseline on Green View Index")
print("-" * 50)
print(f"  Threshold (GVI):    {baseline['threshold']:.4f}")
print(f"  AUC:                {baseline['auc']:.4f}")
print(f"  Average Precision:  {baseline['average_precision']:.4f}")
print(
    f"  Happy above thr.:   {int(baseline['happy_above_threshold'])}/"
    f"{int(baseline['happy_total'])}"
)

**Baseline result: AUC = 0.507** — essentially chance level. The single feature (greenery) cannot discriminate happy from non-happy. This is the performance floor against which the primary model must justify its complexity.

### 4.2 Primary Model: Two-Stage PU Learning

**Stage 1 — Identify reliable negatives.**

1. Standardize the 14-feature matrix with `StandardScaler`.
2. Compute the centroid of the 28 positive samples in standardized space.
3. Compute Euclidean distance from each unlabeled point to the centroid.
4. Select the **30% farthest** unlabeled points (~13,027 points) as *reliable negatives*.
5. Track the closest 5% as *suspect positives* for post-hoc inspection (candidate overlooked happy places).

**Stage 2 — Train classifiers.**

- **Logistic Regression** with `class_weight='balanced'`, `max_iter=1000`, 14 input features. Prioritized for interpretability — coefficients quantify each feature's direction and magnitude.
- **Random Forest** with `n_estimators=200`, `max_depth=10`, `min_samples_leaf=5`, `class_weight='balanced'`. Prioritized for predictive performance and to capture non-linear interactions.
- **Stratified 5-fold cross-validation** with `random_state=42`. Each fold contains ~22–23 training positives and 5–6 validation positives.

**Why not deep learning?** 28 positive samples is far below the regime where CNNs or transformers can generalize. The two-stage pipeline is more robust under extreme label scarcity, and LR coefficients provide policy-relevant interpretability that deep models cannot match.

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler


class PULearning:
    """Two-stage PU Learning: distance-based reliable negatives + LR/RF."""

    def __init__(self, reliable_neg_ratio=0.30):
        self.reliable_neg_ratio = reliable_neg_ratio
        self.scaler = StandardScaler()

    def fit(self, X_positive, X_unlabeled):
        self.scaler.fit(np.vstack([X_positive, X_unlabeled]))
        X_pos = self.scaler.transform(X_positive)
        X_unl = self.scaler.transform(X_unlabeled)

        centroid = X_pos.mean(axis=0)
        distances = cdist(X_unl, [centroid]).flatten()
        n_neg = int(len(X_unl) * self.reliable_neg_ratio)
        self.neg_idx = np.argsort(distances)[-n_neg:]

        self.X_train = np.vstack([X_pos, X_unl[self.neg_idx]])
        self.y_train = np.concatenate([np.ones(len(X_pos)), np.zeros(n_neg)])

        self.lr = LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=42
        ).fit(self.X_train, self.y_train)
        self.rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ).fit(self.X_train, self.y_train)
        return self

    def cv_auc(self):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        lr = cross_val_score(
            self.lr, self.X_train, self.y_train, cv=cv, scoring="roc_auc"
        )
        rf = cross_val_score(
            self.rf, self.X_train, self.y_train, cv=cv, scoring="roc_auc"
        )
        return (lr.mean(), lr.std()), (rf.mean(), rf.std())

### 4.3 Comparison: Is the Added Complexity Worth the Cost?

| Model | ROC-AUC | Complexity | Interpretability |
|-------|---------|------------|------------------|
| Otsu baseline (1 feature) | **0.507** | Minimal | Single threshold |
| Logistic Regression (14 features) | **0.968 ± 0.022** | Moderate | Full coefficients |
| Random Forest (14 features) | **0.979 ± 0.017** | High | Mean decrease in impurity |

The primary model improves AUC by **+0.47** over the baseline — the difference between "completely random" and "near-perfect discrimination." That gap is driven almost entirely by integrating socioeconomic context with visual features, not by model sophistication: LR and RF are within 0.01 of each other, so the *feature space* earns the lift, not the algorithm. The added complexity is clearly worth the cost.

---

## 5. Evaluation & Analysis

### 5.1 Primary Metric: ROC-AUC with 5-fold CV

Under extreme class imbalance (28 positive / 13,027 negative in training), accuracy-based metrics are severely distorted. ROC-AUC is robust to imbalance, threshold-independent, and directly comparable across models. We report mean ± 1 σ across five stratified folds.

In [ ]:
from IPython.display import Image, display

display(Image(filename="final_project_files/4.visualization/fig1_roc_curve.png"))

**Figure 1 reading.** Both LR and RF curves hug the top-left corner. The shaded band is ±1 σ across folds — narrow, confirming stability despite only 5–6 validation positives per fold. The Otsu baseline tracks the diagonal (AUC ≈ chance). Both primary models exceed the proposal's 0.85 threshold for planning decision support by a wide margin.

### 5.2 Supplementary Metric: Precision-Recall

The 28:13,027 imbalance makes ROC optimistic-looking by construction; PR is the stricter test.

In [ ]:
display(Image(filename="final_project_files/4.visualization/fig2_precision_recall.png"))

**Figure 2 reading.** The "No Skill" baseline (P = 0.002) reflects the training-set positive rate. Both LR and RF maintain precision far above this line across nearly the full recall range, confirming that high AUC is not an artifact of the class-imbalance inflating ROC. The Otsu PR curve sits near the no-skill line, consistent with its AUC ≈ 0.5.

### 5.3 Feature Importance and Coefficient Significance

LR coefficients are reported with standard errors computed via the Wald test (Hessian-based). Significance: `*` p<0.05, `**` p<0.01, `***` p<0.001.

In [ ]:
importance = pd.read_csv(DATA / "modeling_results" / "feature_importance_pu.csv")
cols = [
    "feature",
    "lr_coefficient",
    "lr_std_error",
    "lr_p_value",
    "significance",
    "rf_importance",
]
importance[cols].round(4)

In [ ]:
display(Image(filename="final_project_files/4.visualization/fig3_feature_importance.png"))

**Figure 3 reading.**

*What the model gets right:*
- `pct_owner_occupied` is the strongest negative predictor (β=−2.06, ***). Happy places cluster in mixed-use, walkable, rental-heavy neighborhoods rather than quiet owner-occupied residential streets — the **owner-occupancy paradox**. This is counterintuitive but consistent with the location set (Rittenhouse, University City, Fairmount).
- `road_ratio` is strongly negative (β=−1.25, ***), and `building_ratio` is strongly positive (β=+1.06, ***). Dense, car-light environments align with happy locations — supporting the "livable streets" and 15-minute-city literature.
- `sky_ratio` positive (β=+0.53, ***) alongside `building_ratio` positive suggests happiness concentrates in **moderately dense** environments: enough urban vitality to be interesting, not so much as to feel oppressive.

*What the model gets wrong (or at least surprising):*
- `green_view_index` is **not significant** (β=−0.04, p=0.44). The urban-perception literature strongly predicts greenery should matter. We interpret this as a manifestation of the hidden-garden problem: several happy places are rooftop parks, internal courtyards, or tree-shaded interiors that road-level GSV cannot see.
- `pct_white` positive (β=+0.52, ***) reflects sampling bias in the original survey (predominantly undergraduate respondents at Drexel and Penn concentrated in Center City / University City), not a generalizable predictor.
- `median_home_value` has a **negative** coefficient (β=−0.31, ***) despite happy places being in higher-home-value tracts on average — a classic multicollinearity-induced direction flip after controlling for `pct_owner_occupied`, `pct_white`, and `pct_college`. In the univariate *t*-test happy points' mean home value is $400k vs. $244k for others, but the multivariate effect attributed specifically to home value (holding other covariates fixed) is slightly negative.

### 5.4 City-Wide Scoring Distribution

In [ ]:
display(Image(filename="final_project_files/4.visualization/fig4_score_distribution.png"))

In [ ]:
pred = pd.read_csv(DATA / "modeling_results" / "happiness_predictions_pu.csv")
stats = (
    pred.groupby("sample_type")["happiness_score"]
    .agg(["count", "mean", "median"])
    .round(4)
)
stats

**Figure 4 reading.**

- **Reliable negatives** (n=13,027) cluster near 0 — the model is confident these are *not* happy places, which is consistent with how they were selected (farthest from the positive centroid).
- **Unlabeled** points (n=28,228) show a right-skewed distribution peaking near 0.1 — most of the city is unlikely to resemble happy places, which matches intuition that happy places are *special*, not the norm.
- **Suspect positives** (n=2,171) cluster at scores 0.7–1.0 — these are the most actionable output for planners: unlabeled points whose features most resemble confirmed happy places and which warrant fieldwork to validate or investigate.
- **Labeled positives** (n=28) are pinned at 1.0 by construction.

### 5.5 Limitations

- **Sample size.** 28 positives bound what we can learn; effects smaller than roughly β ≈ 0.2 are lost in noise.
- **Sampling bias.** Respondents were Drexel undergraduates. Predictions are most reliable for places valued by that demographic; least reliable for locations meaningful to older, lower-income, or non-student populations.
- **Hidden gardens.** Several happy places cannot be directly photographed. The features assigned to those points reflect adjacent streets, not the places themselves.
- **Ecological fallacy.** Tract-level Census assigned to individual street points assumes within-tract homogeneity. Tracts with ~4,100 residents can span significant socioeconomic variation.
- **Temporal blindness.** Static GSV imagery ignores seasonal (leaf-off vs. leaf-on) and diurnal variation. A place that feels vibrant on a Saturday afternoon may not on a Tuesday morning.
- **Correlation, not causation.** We cannot claim dense buildings *cause* happiness; the model identifies spatial co-occurrence patterns.

---

## 6. Future Work

**Expand the positive set.** 28 points is the binding constraint. A city-wide crowdsourced happiness mapping campaign (app-based, social-media mining, or follow-up surveys to other universities and community groups) could yield hundreds or thousands of positives, broaden the demographic representation, and reduce variance.

**Temporal analysis.** Replace single-snapshot GSV with multi-season imagery. Seasonal green view index changes could resolve whether greenery's weak effect is real or an artifact of imagery mixing winter and summer captures.

**Sky View Factor.** Replace `sky_ratio` (a raw pixel proportion) with a proper hemispherical Sky View Factor that weights overhead sky more than horizon sky, matching human perception of openness. Li & Ratti's Otsu-on-fisheye approach is directly applicable.

**Spatial cross-validation.** The current 5-fold CV ignores spatial autocorrelation. Happy places cluster geographically; IID folds may overestimate generalization. Block- or region-based CV would give a more honest estimate of how the model performs in *new* neighborhoods.

**Qualitative validation.** Interview a sample of the original Drexel respondents — or, better, visit the top 50 suspect-positive points and assess in person whether they "feel" happy. This grounds quantitative patterns in human experience and can surface biases the model cannot self-detect.

**Operationalization.** Publish the happiness surface as a WMS/WMTS layer on OpenDataPhilly. Pair it with a simple decision-support tool: planners enter a proposed intervention location (new park, road diet, façade program), and the tool reports the predicted happiness delta using counterfactual feature edits. This closes the loop from academic scoring to planning practice.

**Multi-city transfer.** Test whether the 14-feature model trained on Philadelphia transfers to Boston, Pittsburgh, or Baltimore. If coefficients are stable across cities, the method generalizes; if not, the happiness signal is city-specific and the model needs local retraining.

---

## 7. Conclusion: What We Learned, and What We Couldn't

Urban happiness is **measurable, and partially predictable, from environmental features**. Sky visibility and building density correlate positively with happiness; road coverage correlates negatively. High-homeownership neighborhoods are *less* likely to contain happiness points, suggesting "livability" and "happiness" are distinct concepts that urban planning has historically confused. Education and income predict happiness; poverty predicts its absence. Our Logistic Regression reaches AUC = 0.968 and Random Forest reaches 0.979 — both well above the 0.85 threshold we set for planning decision support, and both dwarfing the Otsu baseline's 0.507. The 14-dimensional feature space earns those numbers; the algorithm choice barely matters inside it.

That is where the honest part begins.

The numbers only tell part of the story. Some happy places are hidden from street-view cameras — a rooftop park, an inner courtyard, a bench tucked behind a community garden. Some happiness comes from memory, not environment — a specific kitchen, a specific Sunday, a specific conversation you've replayed for years. Some of our variation is seasonal, daily, personal, and irreducibly private.

This project demonstrates how street-level imagery and machine learning can be combined to study urban emotion at scale. It extends psychological theories of place and well-being into a computational framework, transforming visual data into measurable indicators of how cities make people feel. But we should be humble about what we've actually captured. Everyone living in Philadelphia has their own list of happy places, and there is never a single right answer to what happiness means. The cat park in Mantua with its friendly felines, the bustling atmosphere of Mango Mango Desserts, the cozy corner of Maison Sweet where you might stay longer than planned — these are not places we can fully quantify. A grandmother's kitchen. A spot by the river where someone proposed. The bench where you sat crying after a breakup and a stranger offered you a tissue.

Happiness is environmental, yes. But it is also biographical. Our models can find the patterns that connect happy places across a city; they cannot explain why a particular place became *your* happy place.

What we hope is that by mapping these patterns, we might help create more spaces where happiness can happen — even if we cannot predict exactly who will find joy there, or why.

**A lot of things need fixing in Philly, but there are a lot of good things here.**

---

## References

- Li, X., & Ratti, C. (2019). Using Google Street View for street-level urban form analysis, a case study in Cambridge, Massachusetts. In L. D'Acci (Ed.), *The Mathematics of Urban Morphology* (pp. 457–470). Springer.
- Li, X., Zhang, C., Li, W., Ricard, R., Meng, Q., & Zhang, W. (2015). Assessing street-level urban greenery using Google Street View and a modified green view index. *Urban Forestry & Urban Greening, 14*(3), 675–685.
- Naik, N., Kominers, S. D., Raskar, R., Glaeser, E. L., & Hidalgo, C. A. (2017). Computer vision uncovers predictors of physical urban change. *Proceedings of the National Academy of Sciences, 114*(29), 7571–7576.
- Xie, E., Wang, W., Yu, Z., Anandkumar, A., Alvarez, J. M., & Luo, P. (2021). SegFormer: Simple and efficient design for semantic segmentation with transformers. *Advances in Neural Information Processing Systems, 34*.
- Zillmer, E. (2025, June 25). Philly psychology students map out local landmarks and hidden destinations where they feel happiest. *The Conversation*. https://doi.org/10.64628/AAI.e5fagj6ff